# Import dependencies

In [ ]:
import pandas as pd
import numpy as np

# Data

## Load resale data

In [ ]:
resale = pd.read_csv("Data/Resale_with_Coords.csv")

In [ ]:
print(f"Shape: {resale.shape}")
print(resale.info())
resale.describe(include='all').T

Shape: (225127, 14)
<class 'pandas.DataFrame'>
RangeIndex: 225127 entries, 0 to 225126
Data columns (total 14 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   month                225127 non-null  str    
 1   town                 225127 non-null  str    
 2   flat_type            225127 non-null  str    
 3   block                225127 non-null  str    
 4   street_name          225127 non-null  str    
 5   storey_range         225127 non-null  str    
 6   floor_area_sqm       225127 non-null  float64
 7   flat_model           225127 non-null  str    
 8   lease_commence_date  225127 non-null  int64  
 9   remaining_lease      225127 non-null  str    
 10  resale_price         225127 non-null  float64
 11  address              225127 non-null  str    
 12  latitude             225127 non-null  float64
 13  longitude            225127 non-null  float64
dtypes: float64(4), int64(1), str(9)
memory usage: 24.0 MB
None


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
month,225127,110,2024-07,3036,NaN,NaN,NaN,NaN,NaN,NaN,NaN
town,225127,26,SENGKANG,18391,NaN,NaN,NaN,NaN,NaN,NaN,NaN
flat_type,225127,7,4 ROOM,95464,NaN,NaN,NaN,NaN,NaN,NaN,NaN
block,225127,2749,2,682,NaN,NaN,NaN,NaN,NaN,NaN,NaN
street_name,225127,577,YISHUN RING RD,3208,NaN,NaN,NaN,NaN,NaN,NaN,NaN
storey_range,225127,17,04 TO 06,51639,NaN,NaN,NaN,NaN,NaN,NaN,NaN
floor_area_sqm,225127.0,NaN,NaN,NaN,96.751932,24.019493,31.0,81.0,93.0,112.0,366.7
flat_model,225127,21,Model A,80573,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lease_commence_date,225127.0,NaN,NaN,NaN,1996.463969,14.314362,1966.0,1985.0,1997.0,2012.0,2021.0
remaining_lease,225127,696,94 years 10 months,1919,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Missing Data Check [None]

In [ ]:
pd.DataFrame({
    'nunique': resale.nunique(dropna=False),
    'missing': resale.isna().sum(),
}).sort_values('nunique', ascending=False)

,nunique,missing
address,9684,0
latitude,9683,0
longitude,9683,0
resale_price,4590,0
block,2749,0
remaining_lease,696,0
street_name,577,0
floor_area_sqm,187,0
month,110,0
lease_commence_date,56,0


In [ ]:
resale = resale.dropna()

## Data Description

In [ ]:
# 'role': 'feature', 'target', 'identifier', 'metadata', 'ambiguous'
# 'type':  'num-cont', 'num-disc', 'cat-nom', 'cat-ord', 'bool'
role_map = {
    "month": "feature",
    "town": "feature",
    "flat_type": "feature",
    "block": "identifier",
    "street_name": "identifier",
    "storey_range": "feature",
    "floor_area_sqm": "feature",
    "flat_model": "metadata",
    "lease_commence_date": "metadata",
    "remaining_lease": "feature",
    "resale_price": "target"
}
type_map = {
    "month": "num-disc",
    "town": "cat-nom",
    "flat_type": "cat-nom",
    "block": "cat-nom",
    "street_name": "cat-nom",
    "storey_range": "cat-ord",
    "floor_area_sqm": "num-disc",
    "flat_model": "cat-nom",
    "lease_commence_date": "num-disc",
    "remaining_lease": "num-disc",
    "resale_price": "num-disc"
}

data_description = pd.DataFrame({
    "column": resale.columns,
})
data_description["role"] = data_description["column"].map(role_map).fillna("unknown")
data_description["type"] = data_description["column"].map(type_map).fillna("unknown")
data_description

,column,role,type
0,month,feature,num-disc
1,town,feature,cat-nom
2,flat_type,feature,cat-nom
3,block,identifier,cat-nom
4,street_name,identifier,cat-nom
5,storey_range,feature,cat-ord
6,floor_area_sqm,feature,num-disc
7,flat_model,metadata,cat-nom
8,lease_commence_date,metadata,num-disc
9,remaining_lease,feature,num-disc


## Data Pre-Processing

### Model Dataset Descriptions

In [ ]:
model_data_desc = data_description[data_description['role'] != "metadata"]
model_data_desc

,column,role,type
0,month,feature,num-disc
1,town,feature,cat-nom
2,flat_type,feature,cat-nom
3,block,identifier,cat-nom
4,street_name,identifier,cat-nom
5,storey_range,feature,cat-ord
6,floor_area_sqm,feature,num-disc
9,remaining_lease,feature,num-disc
10,resale_price,target,num-disc
11,address,unknown,unknown


### Model Dataset

In [ ]:
model_data = resale[model_data_desc["column"]].copy()
model_data.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,remaining_lease,resale_price,address,latitude,longitude
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,61 years 04 months,232000.0,406 ANG MO KIO AVE 10,1.362005,103.853880
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,60 years 07 months,250000.0,108 ANG MO KIO AVE 4,1.370966,103.838202
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,62 years 05 months,262000.0,602 ANG MO KIO AVE 5,1.380709,103.835368
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,62 years 01 month,265000.0,465 ANG MO KIO AVE 10,1.366201,103.857201
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,62 years 05 months,265000.0,601 ANG MO KIO AVE 5,1.381041,103.835132


### Data Transformation

# Feature Extraction

Add amenity count for the varies amenities (mrt,bus-stop, schools, etc..)

## Import Dependencies

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

## Load Data

In [ ]:
mrt_df = pd.read_csv("Data/mrt_stations.csv")
schools_df = pd.read_csv("Data/schools.csv")
malls_df = pd.read_csv("Data/shopping_malls.csv")
bus_stop_df = pd.read_csv("Data/bus_stops.csv")
hawker_df = pd.read_csv("Data/hawker_centres.csv")
wet_market_df = pd.read_csv("Data/wet_markets.csv")

## Project Data on Point

In [ ]:
mrt_gdf = gpd.GeoDataFrame(mrt_df, geometry=gpd.points_from_xy(mrt_df.long, mrt_df.lat), crs="EPSG:4326")
schools_gdf = gpd.GeoDataFrame(schools_resale, geometry=gpd.points_from_xy(schools_resale.long, schools_resale.lat),
                               crs="EPSG:4326")
malls_gresale = gpd.GeoDataFrame(malls_resale, geometry=gpd.points_from_xy(malls_resale.long, malls_resale.lat), crs="EPSG:4326")
bus_stop_gdf = gpd.GeoDataFrame(bus_stop_df, geometry=gpd.points_from_xy(bus_stop_df.long, bus_stop_df.lat),
                                crs="EPSG:4326")
hawker_gdf = gpd.GeoDataFrame(hawker_df, geometry=gpd.points_from_xy(hawker_df.long, hawker_df.lat), crs="EPSG:4326")
wet_market_gdf = gpd.GeoDataFrame(wet_market_df, geometry=gpd.points_from_xy(wet_market_df.long, wet_market_df.lat),
                                  crs="EPSG:4326")

In [ ]:
flat_gdf = gpd.GeoDataFrame(transformed, geometry=gpd.points_from_xy(transformed.longitude, transformed.latitude),
                            crs="EPSG:4326")
flat_gdf.head()

,month,town,flat_type,floor_area_sqm,remaining_lease,resale_price,latitude,longitude,storey_type_lower,storey_type_middle,storey_type_upper,rpi,adjusted_resale,geometry
0,0,ANG MO KIO,2 ROOM,44.0,736,232000.0,1.362005,103.853880,False,False,True,133.9,173263.629574,POINT (103.85388 1.362)
1,0,ANG MO KIO,3 ROOM,67.0,727,250000.0,1.370966,103.838202,True,False,False,133.9,186706.497386,POINT (103.8382 1.37097)
2,0,ANG MO KIO,3 ROOM,67.0,749,262000.0,1.380709,103.835368,True,False,False,133.9,195668.409261,POINT (103.83537 1.38071)
3,0,ANG MO KIO,3 ROOM,68.0,745,265000.0,1.366201,103.857201,False,True,False,133.9,197908.887229,POINT (103.8572 1.3662)
4,0,ANG MO KIO,3 ROOM,67.0,749,265000.0,1.381041,103.835132,True,False,False,133.9,197908.887229,POINT (103.83513 1.38104)


In [ ]:
# Reproject to a Metric CRS (Singapore SVY21 is EPSG:3414)
flat_gdf = flat_gdf.to_crs(epsg=3414)
mrt_gdf = mrt_gdf.to_crs(epsg=3414)
schools_gdf = schools_gdf.to_crs(epsg=3414)
malls_gdf = malls_gdf.to_crs(epsg=3414)
bus_stop_gdf = bus_stop_gdf.to_crs(epsg=3414)
hawker_gdf = hawker_gdf.to_crs(epsg=3414)
wet_market_gdf = wet_market_gdf.to_crs(epsg=3414)

## Create Buffer zone

In [ ]:
# Create a 500m buffer around each house
flat_gdf['geometry'] = flat_gdf.geometry.buffer(500)

## Get Nearby amenties in radius (500M) (New Feature columns)

In [ ]:
# Spatial Join: Count MRT stations inside the house buffers
joined = gpd.sjoin(flat_gdf, mrt_gdf, how="left", predicate="intersects")
school_joined = gpd.sjoin(flat_gdf, schools_gdf, how="left", predicate="intersects")
mall_joined = gpd.sjoin(flat_gdf, malls_gdf, how="left", predicate="intersects")
bus_stop_joined = gpd.sjoin(flat_gdf, bus_stop_gdf, how="left", predicate="intersects")
hawker_joined = gpd.sjoin(flat_gdf, hawker_gdf, how="left", predicate="intersects")
wet_market_joined = gpd.sjoin(flat_gdf, wet_market_gdf, how="left", predicate="intersects")

mrt_counts = joined.groupby(joined.index).size() - joined['index_right'].isna().groupby(joined.index).sum()
school_counts = school_joined.groupby(school_joined.index).size() - school_joined['index_right'].isna().groupby(
    school_joined.index).sum()
mall_counts = mall_joined.groupby(mall_joined.index).size() - mall_joined['index_right'].isna().groupby(
    mall_joined.index).sum()
bus_stop_counts = bus_stop_joined.groupby(bus_stop_joined.index).size() - bus_stop_joined['index_right'].isna().groupby(
    bus_stop_joined.index).sum()
hawker_counts = hawker_joined.groupby(hawker_joined.index).size() - hawker_joined['index_right'].isna().groupby(
    hawker_joined.index).sum()
wet_market_counts = wet_market_joined.groupby(wet_market_joined.index).size() - wet_market_joined[
    'index_right'].isna().groupby(wet_market_joined.index).sum()

transformed['mrt_count'] = mrt_counts.values
transformed['school_count'] = school_counts.values
transformed['mall_count'] = mall_counts.values
transformed['bus_stop_count'] = bus_stop_counts.values
transformed['hawker_count'] = hawker_counts.values
transformed['wet_market_count'] = wet_market_counts.values
transformed.head()

,month,town,flat_type,floor_area_sqm,remaining_lease,resale_price,latitude,longitude,storey_type_lower,storey_type_middle,storey_type_upper,rpi,adjusted_resale,mrt_count,school_count,mall_count,bus_stop_count,hawker_count,wet_market_count
0,0,ANG MO KIO,2 ROOM,44.0,736,232000.0,1.362005,103.853880,False,False,True,133.9,173263.629574,0,3,0,14,1,0
1,0,ANG MO KIO,3 ROOM,67.0,727,250000.0,1.370966,103.838202,True,False,False,133.9,186706.497386,1,2,0,11,1,0
2,0,ANG MO KIO,3 ROOM,67.0,749,262000.0,1.380709,103.835368,True,False,False,133.9,195668.409261,1,0,0,12,0,0
3,0,ANG MO KIO,3 ROOM,68.0,745,265000.0,1.366201,103.857201,False,True,False,133.9,197908.887229,0,1,0,10,2,0
4,0,ANG MO KIO,3 ROOM,67.0,749,265000.0,1.381041,103.835132,True,False,False,133.9,197908.887229,1,0,0,11,0,0


## Calculate distance to nearest bus-stop and MRT station in meters (New Feature Columns)

In [ ]:
# Create GeoDataFrame from transformed data
flat_point_gdf = gpd.GeoDataFrame(
    transformed,
    geometry=gpd.points_from_xy(transformed.longitude, transformed.latitude),
    crs="EPSG:4326"
)
flat_point_gdf = flat_point_gdf.to_crs(epsg=3414)
print("✓ flat_point_gdf created and reprojected.")

# --- Calculate Distance to Nearest MRT ---
print("Calculating nearest MRT distances...")
mrt_distances = []

for idx, flat in flat_point_gdf.iterrows():
    distances_to_all_mrt = mrt_gdf.geometry.distance(flat.geometry)
    min_distance = distances_to_all_mrt.min()
    mrt_distances.append(min_distance)

transformed['nearest_mrt_dist'] = pd.Series(mrt_distances).round(0).astype(int)
print(f"✓ Nearest MRT distance calculated for {len(mrt_distances)} flats.")

# --- Calculate Distance to Nearest Bus Stop ---
print("Calculating nearest bus stop distances...")
bus_distances = []

for idx, flat in flat_point_gdf.iterrows():
    distances_to_all_bus = bus_stop_gdf.geometry.distance(flat.geometry)
    min_distance = distances_to_all_bus.min()
    bus_distances.append(min_distance)

transformed['nearest_bus_stop_dist'] = pd.Series(bus_distances).round(0).astype(int)
print(f"✓ Nearest bus stop distance calculated for {len(bus_distances)} flats.")

# --- Validation ---
print(f"\nValidation:")
print(f"  MRT - NaN values: {transformed['nearest_mrt_dist'].isna().sum()}")
print(f"  Bus - NaN values: {transformed['nearest_bus_stop_dist'].isna().sum()}")
print(f"  MRT - Mean distance: {transformed['nearest_mrt_dist'].mean():.0f}m")
print(f"  Bus - Mean distance: {transformed['nearest_bus_stop_dist'].mean():.0f}m")

display(transformed.head())

✓ flat_point_gdf created and reprojected.
Calculating nearest MRT distances...
✓ Nearest MRT distance calculated for 225127 flats.
Calculating nearest bus stop distances...
✓ Nearest bus stop distance calculated for 225127 flats.

Validation:
  MRT - NaN values: 0
  Bus - NaN values: 0
  MRT - Mean distance: 565m
  Bus - Mean distance: 113m


,month,town,flat_type,floor_area_sqm,remaining_lease,resale_price,latitude,longitude,storey_type_lower,storey_type_middle,...,rpi,adjusted_resale,mrt_count,school_count,mall_count,bus_stop_count,hawker_count,wet_market_count,nearest_mrt_dist,nearest_bus_stop_dist
0,0,ANG MO KIO,2 ROOM,44.0,736,232000.0,1.362005,103.853880,False,False,...,133.9,173263.629574,0,3,0,14,1,0,1016,91
1,0,ANG MO KIO,3 ROOM,67.0,727,250000.0,1.370966,103.838202,True,False,...,133.9,186706.497386,1,2,0,11,1,0,202,164
2,0,ANG MO KIO,3 ROOM,67.0,749,262000.0,1.380709,103.835368,True,False,...,133.9,195668.409261,1,0,0,12,0,0,460,136
3,0,ANG MO KIO,3 ROOM,68.0,745,265000.0,1.366201,103.857201,False,True,...,133.9,197908.887229,0,1,0,10,2,0,828,68
4,0,ANG MO KIO,3 ROOM,67.0,749,265000.0,1.381041,103.835132,True,False,...,133.9,197908.887229,1,0,0,11,0,0,433,146


In [ ]:
import numpy as np

transformed['nearest_mrt_dist'] = np.floor(transformed['nearest_mrt_dist'])
transformed['nearest_bus_stop_dist'] = np.floor(transformed['nearest_bus_stop_dist'])

display(transformed.head())

,month,town,flat_type,floor_area_sqm,remaining_lease,resale_price,latitude,longitude,storey_type_lower,storey_type_middle,...,rpi,adjusted_resale,mrt_count,school_count,mall_count,bus_stop_count,hawker_count,wet_market_count,nearest_mrt_dist,nearest_bus_stop_dist
0,0,ANG MO KIO,2 ROOM,44.0,736,232000.0,1.362005,103.853880,False,False,...,133.9,173263.629574,0,3,0,14,1,0,1016,91
1,0,ANG MO KIO,3 ROOM,67.0,727,250000.0,1.370966,103.838202,True,False,...,133.9,186706.497386,1,2,0,11,1,0,202,164
2,0,ANG MO KIO,3 ROOM,67.0,749,262000.0,1.380709,103.835368,True,False,...,133.9,195668.409261,1,0,0,12,0,0,460,136
3,0,ANG MO KIO,3 ROOM,68.0,745,265000.0,1.366201,103.857201,False,True,...,133.9,197908.887229,0,1,0,10,2,0,828,68
4,0,ANG MO KIO,3 ROOM,67.0,749,265000.0,1.381041,103.835132,True,False,...,133.9,197908.887229,1,0,0,11,0,0,433,146


The `nearest_mrt_dist` and `nearest_bus_stop_dist` columns have now been rounded down to the nearest whole number. You can see the updated values in the displayed DataFrame.

**Month** \
Convert to indexing for model input, index represents the order for time-series models \
index starts from 2017-01 onwards \
Example:

| Month(before) | index(after) |
| ------ | ------ |
| 2017-01 | 0 |
| 2017-02 | 1 |

In [ ]:
months = pd.to_datetime(model_data['month'], format='%Y-%m')
model_data['month'] = (
        months.dt.year * 12 + months.dt.month
)

model_data['month'] -= model_data['month'].min()
model_data.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,remaining_lease,resale_price,address,latitude,longitude
0,0,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,61 years 04 months,232000.0,406 ANG MO KIO AVE 10,1.362005,103.853880
1,0,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,60 years 07 months,250000.0,108 ANG MO KIO AVE 4,1.370966,103.838202
2,0,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,62 years 05 months,262000.0,602 ANG MO KIO AVE 5,1.380709,103.835368
3,0,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,62 years 01 month,265000.0,465 ANG MO KIO AVE 10,1.366201,103.857201
4,0,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,62 years 05 months,265000.0,601 ANG MO KIO AVE 5,1.381041,103.835132


**remaining_lease [Months]** \
Convert remaining_lease to number of months instead of X years X months

In [ ]:
remaining_yrs = model_data['remaining_lease'].str.extract(r'(\d+)\D+(?:(\d+)\D+)?').fillna(0).astype(int)
model_data['remaining_lease'] = remaining_yrs[0] * 12 + remaining_yrs[1]
model_data.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,remaining_lease,resale_price,address,latitude,longitude
0,0,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,736,232000.0,406 ANG MO KIO AVE 10,1.362005,103.853880
1,0,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,727,250000.0,108 ANG MO KIO AVE 4,1.370966,103.838202
2,0,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,749,262000.0,602 ANG MO KIO AVE 5,1.380709,103.835368
3,0,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,745,265000.0,465 ANG MO KIO AVE 10,1.366201,103.857201
4,0,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,749,265000.0,601 ANG MO KIO AVE 5,1.381041,103.835132


**Storey type** \
Replace storey range with the storey type (lower, middle, upper)

In [ ]:
avg_storey = model_data['storey_range'].str.split(" TO ", expand=True).astype(int).mean(axis=1)
model_data['storey_type'] = pd.cut(avg_storey, bins=[1, 3, 7, 99], labels=['lower', 'middle', 'upper'], right=False)
model_data = pd.get_dummies(model_data, columns=['storey_type'])
model_data.drop(columns=['storey_range'], inplace=True)
model_data.head()

,month,town,flat_type,block,street_name,floor_area_sqm,remaining_lease,resale_price,address,latitude,longitude,storey_type_lower,storey_type_middle,storey_type_upper
0,0,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,44.0,736,232000.0,406 ANG MO KIO AVE 10,1.362005,103.853880,False,False,True
1,0,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,67.0,727,250000.0,108 ANG MO KIO AVE 4,1.370966,103.838202,True,False,False
2,0,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,67.0,749,262000.0,602 ANG MO KIO AVE 5,1.380709,103.835368,True,False,False
3,0,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,68.0,745,265000.0,465 ANG MO KIO AVE 10,1.366201,103.857201,False,True,False
4,0,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,67.0,749,265000.0,601 ANG MO KIO AVE 5,1.381041,103.835132,True,False,False


**Quarter index** \
Defines which quarter is the data in example 2 for 2017-04 to 2017-07

In [ ]:
model_data['quarter'] = model_data['month'] // 3
model_data.tail()

,month,town,flat_type,block,street_name,floor_area_sqm,remaining_lease,resale_price,address,latitude,longitude,storey_type_lower,storey_type_middle,storey_type_upper,quarter
225122,108,YISHUN,EXECUTIVE,325,YISHUN CTRL,146.0,743,920000.0,325 YISHUN CTRL,1.429239,103.842146,False,True,False,36
225123,108,YISHUN,EXECUTIVE,360,YISHUN RING RD,142.0,739,865888.0,360 YISHUN RING RD,1.427737,103.845686,False,False,True,36
225124,108,YISHUN,EXECUTIVE,643,YISHUN ST 61,142.0,729,825000.0,643 YISHUN ST 61,1.421335,103.837437,False,False,True,36
225125,108,YISHUN,EXECUTIVE,643,YISHUN ST 61,146.0,728,788000.0,643 YISHUN ST 61,1.421335,103.837437,False,True,False,36
225126,109,YISHUN,EXECUTIVE,611,YISHUN ST 61,146.0,730,860088.0,611 YISHUN ST 61,1.420201,103.836153,False,True,False,36


**Resale Price Index [RPI]**

In [ ]:
rpi = pd.read_csv('Data/2025-RPI.csv')
rpi = rpi[rpi['year'] >= 2017]
rpi['quarter'] = (rpi['year'] - 2017) * 4 + (rpi['quarter'] - 1)
rpi.drop(columns=['year'], inplace=True)
rpi.head()

,quarter,rpi
32,0,133.9
33,1,133.7
34,2,132.8
35,3,132.6
36,4,131.6


In [ ]:
model_data = pd.merge(model_data, rpi, on='quarter', how='left')
model_data.head()

,month,town,flat_type,block,street_name,floor_area_sqm,remaining_lease,resale_price,address,latitude,longitude,storey_type_lower,storey_type_middle,storey_type_upper,quarter,rpi
0,0,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,44.0,736,232000.0,406 ANG MO KIO AVE 10,1.362005,103.853880,False,False,True,0,133.9
1,0,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,67.0,727,250000.0,108 ANG MO KIO AVE 4,1.370966,103.838202,True,False,False,0,133.9
2,0,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,67.0,749,262000.0,602 ANG MO KIO AVE 5,1.380709,103.835368,True,False,False,0,133.9
3,0,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,68.0,745,265000.0,465 ANG MO KIO AVE 10,1.366201,103.857201,False,True,False,0,133.9
4,0,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,67.0,749,265000.0,601 ANG MO KIO AVE 5,1.381041,103.835132,True,False,False,0,133.9


**Adjusted Resale Price** \
Formula: Resale price / RPI

In [ ]:
model_data['adjusted_resale'] = model_data['resale_price'] / (model_data['rpi'] / 100)
model_data.head()

,month,town,flat_type,block,street_name,floor_area_sqm,remaining_lease,resale_price,address,latitude,longitude,storey_type_lower,storey_type_middle,storey_type_upper,quarter,rpi,adjusted_resale
0,0,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,44.0,736,232000.0,406 ANG MO KIO AVE 10,1.362005,103.853880,False,False,True,0,133.9,173263.629574
1,0,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,67.0,727,250000.0,108 ANG MO KIO AVE 4,1.370966,103.838202,True,False,False,0,133.9,186706.497386
2,0,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,67.0,749,262000.0,602 ANG MO KIO AVE 5,1.380709,103.835368,True,False,False,0,133.9,195668.409261
3,0,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,68.0,745,265000.0,465 ANG MO KIO AVE 10,1.366201,103.857201,False,True,False,0,133.9,197908.887229
4,0,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,67.0,749,265000.0,601 ANG MO KIO AVE 5,1.381041,103.835132,True,False,False,0,133.9,197908.887229


**Remove unneccessary columns**

In [ ]:
transformed = model_data.drop(columns=['block', 'street_name', 'quarter', 'address'])
transformed.head()

,month,town,flat_type,floor_area_sqm,remaining_lease,resale_price,latitude,longitude,storey_type_lower,storey_type_middle,storey_type_upper,rpi,adjusted_resale
0,0,ANG MO KIO,2 ROOM,44.0,736,232000.0,1.362005,103.853880,False,False,True,133.9,173263.629574
1,0,ANG MO KIO,3 ROOM,67.0,727,250000.0,1.370966,103.838202,True,False,False,133.9,186706.497386
2,0,ANG MO KIO,3 ROOM,67.0,749,262000.0,1.380709,103.835368,True,False,False,133.9,195668.409261
3,0,ANG MO KIO,3 ROOM,68.0,745,265000.0,1.366201,103.857201,False,True,False,133.9,197908.887229
4,0,ANG MO KIO,3 ROOM,67.0,749,265000.0,1.381041,103.835132,True,False,False,133.9,197908.887229


# Export Data

In [ ]:
transformed.to_csv('Data/processed.csv', index=False)